In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q pytorch_pretrained_vit

In [ ]:
import math
import os
import random
import shutil

import cv2
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from PIL import Image
from scipy.interpolate import UnivariateSpline
from torch import nn, Tensor
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torchvision.transforms import Compose as TorchvisionCompose
from torchvision.transforms import Lambda
from torchvision.transforms import functional as TVF
from pytorch_pretrained_vit import ViT
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import log_loss

In [ ]:
ARCHIVE = "/content/drive/MyDrive/datadriven/competition_VfIpjyh.zip"
DATA_DIR = "/content/data"

RANK = 8
NUM_CLASSES = 8
FRAC = 1.0
TEST_SIZE = 0.25
EPOCHS = 10
LR = 1e-3
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
SEED = 1
AUGMENT = True
NORM_MEAN = 0.5
NORM_STD = 0.5

device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name() if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64 if "A100" in gpu_name else 32
NUM_WORKERS = os.cpu_count() or 2
print(f"device: {device} ({gpu_name}), batch_size: {BATCH_SIZE}, workers: {NUM_WORKERS}")

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)
shutil.unpack_archive(ARCHIVE, DATA_DIR)
print(os.listdir(DATA_DIR))

In [ ]:
class _LoRALayer(nn.Module):
    def __init__(self, w: nn.Module, w_a: nn.Module, w_b: nn.Module):
        super().__init__()
        self.w = w
        self.w_a = w_a
        self.w_b = w_b

    def forward(self, x):
        return self.w(x) + self.w_b(self.w_a(x))


class LoRA_ViT(nn.Module):
    TARGETS = [("attn", "proj_q"), ("attn", "proj_v")]

    def __init__(self, vit_model: ViT, r: int, num_classes: int = 0, lora_layer=None):
        super().__init__()
        assert r > 0
        if lora_layer:
            self.lora_layer = lora_layer
        else:
            self.lora_layer = list(range(len(vit_model.transformer.blocks)))
        self.w_As = []
        self.w_Bs = []

        for param in vit_model.parameters():
            param.requires_grad = False

        for t_layer_i, blk in enumerate(vit_model.transformer.blocks):
            if t_layer_i not in self.lora_layer:
                continue
            for sub, name in self.TARGETS:
                parent = getattr(blk, sub)
                linear = getattr(parent, name)
                w_a = nn.Linear(linear.in_features, r, bias=False)
                w_b = nn.Linear(r, linear.out_features, bias=False)
                self.w_As.append(w_a)
                self.w_Bs.append(w_b)
                setattr(parent, name, _LoRALayer(linear, w_a, w_b))

        self.reset_parameters()
        self.lora_vit = vit_model
        if num_classes > 0:
            self.lora_vit.fc = nn.Linear(vit_model.fc.in_features, num_classes)

    def reset_parameters(self):
        for w_A in self.w_As:
            nn.init.kaiming_uniform_(w_A.weight, a=math.sqrt(5))
        for w_B in self.w_Bs:
            nn.init.zeros_(w_B.weight)

    def forward(self, x: Tensor) -> Tensor:
        return self.lora_vit(x)

In [ ]:
# --- Augmentations ported verbatim from the MIPT thesis (transforms_img.py) ---
def adjust_brightness(src, brightness_factor):
    return np.array(TVF.adjust_brightness(Image.fromarray(src), brightness_factor))


def adjust_contrast(src, contrast_factor):
    return np.array(TVF.adjust_contrast(Image.fromarray(src), contrast_factor))


def adjust_hue(img, hue_factor):
    return np.array(TVF.adjust_hue(Image.fromarray(img), hue_factor))


def adjust_gamma(img, gamma, gain=1):
    if gamma < 0:
        raise ValueError("Gamma should be a non-negative real number")
    return np.array(TVF.adjust_gamma(Image.fromarray(img), gamma, gain))


def adjust_saturation(img, factor=0.0):
    return np.array(TVF.adjust_saturation(Image.fromarray(img), factor))


def create_LUT_8UC1(x, y):
    spl = UnivariateSpline(x, y)
    return spl(range(256))


def adjust_temperature(img, factor=1):
    if not (-1 <= factor <= 1):
        raise ValueError(f"{factor} is not in [-1, 1].")
    if factor == 0:
        return img
    inp = [0, 64, 128, 192, 256]
    dest_warm = [0, 70, 140, 210, 256]
    dest_cold = [0, 30, 80, 120, 192]
    abs_factor = abs(factor)
    delta = np.array(dest_warm) - np.array(inp)
    dest_warm = (np.array(inp) + np.array(delta) * abs_factor).tolist()
    delta = np.array(dest_cold) - np.array(inp)
    dest_cold = (np.array(inp) + np.array(delta) * abs_factor).tolist()
    incr_ch_lut = create_LUT_8UC1(inp, dest_warm)
    decr_ch_lut = create_LUT_8UC1(inp, dest_cold)
    if factor > 0:
        incr_ch_lut, decr_ch_lut = decr_ch_lut, incr_ch_lut
    img[:, :, 2] = cv2.LUT(img[:, :, 2], incr_ch_lut).astype(np.uint8)
    img[:, :, 0] = cv2.LUT(img[:, :, 0], decr_ch_lut).astype(np.uint8)
    hsv_lut = incr_ch_lut if factor < 0 else decr_ch_lut
    img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    img_hsv[:, :, 1] = cv2.LUT(img_hsv[:, :, 1], hsv_lut).astype(np.uint8)
    img = cv2.cvtColor(img_hsv, cv2.COLOR_HSV2BGR)
    return img


class RandomHorizontalFlip:
    def __init__(self, p=0.5, tags=("image",)):
        self.p = p
        self.tags = tags

    def __call__(self, sample_dict):
        to_flip = random.random() < self.p
        common_tags = set(self.tags).intersection(set(sample_dict.keys()))
        for tag in common_tags:
            if to_flip:
                sample_dict[tag] = np.fliplr(sample_dict[tag].copy())
        return sample_dict


class ColorJitterCV:
    def __init__(self, brightness=0, contrast=0, saturation=0, hue=0, gamma=0, temp=0, p=0.75, tags=("image",)):
        self.brightness = brightness if self.check_param(brightness) else [max(0, 1 - brightness), 1 + brightness]
        self.contrast = contrast if self.check_param(contrast) else [max(0, 1 - contrast), 1 + contrast]
        self.saturation = saturation if self.check_param(saturation) else [-saturation, saturation]
        self.hue = hue if self.check_param(hue) else [-hue, hue]
        self.gamma = gamma if self.check_param(gamma) else [1 - gamma, 1 + gamma]
        self.temp = temp if self.check_param(temp) else [-temp, temp]
        self.brightness = np.clip(self.brightness, 0, None)
        self.contrast = np.clip(self.contrast, 0, None)
        self.hue = np.clip(self.hue, -0.5, 0.5)
        self.gamma = np.clip(self.gamma, 0.5, 1.5)
        self.temp = np.clip(self.temp, -1, 1)
        self.p = p
        self.tags = tags

    @staticmethod
    def check_param(param):
        return hasattr(param, "__len__") and len(param) == 2

    @staticmethod
    def get_params(brightness, contrast, saturation, hue, gamma, temp):
        transforms = []
        gamma_factor = 1
        if not np.allclose(gamma, 1):
            gain_factor = 1
            gamma_factor = np.clip(random.uniform(gamma[0], gamma[1]), 0.5, 1.5)
            transforms.append(Lambda(lambda img: adjust_gamma(np.array(img), gamma_factor, gain_factor)))
        if not np.allclose(brightness, 1):
            if gamma_factor < 1 and brightness[1] > 1:
                brightness_factor = random.uniform(1, brightness[1])
            elif gamma_factor > 1 and brightness[0] < 1:
                brightness_factor = random.uniform(brightness[0], 1)
            elif gamma_factor == 1:
                brightness_factor = random.uniform(brightness[0], brightness[1])
            else:
                brightness_factor = 1
            transforms.append(Lambda(lambda img: adjust_brightness(img, brightness_factor)))
        if not np.allclose(contrast, 1):
            contrast_factor = random.uniform(contrast[0], contrast[1])
            transforms.append(Lambda(lambda img: adjust_contrast(img, contrast_factor)))
        if not np.allclose(saturation, 0):
            saturation_factor = random.uniform(saturation[0], saturation[1])
            transforms.append(Lambda(lambda img: adjust_saturation(img, saturation_factor)))
        if not np.allclose(temp, 0):
            temp_factor = random.uniform(temp[0], temp[1])
            transforms.append(Lambda(lambda img: adjust_temperature(img, temp_factor)))
        if not np.allclose(hue, 1):
            hue_factor = float(np.clip(random.uniform(hue[0], hue[1]), -0.5, 0.5))
            transforms.append(Lambda(lambda img: adjust_hue(img, hue_factor)))
        random.shuffle(transforms)
        return TorchvisionCompose(transforms)

    def __call__(self, sample_dict):
        transform_func = self.get_params(self.brightness, self.contrast, self.saturation, self.hue, self.gamma, self.temp)
        if np.random.random() < self.p:
            for tag in self.tags:
                sample_dict[tag] = transform_func(sample_dict[tag].squeeze())
        return sample_dict


class RandomGaussianBlur:
    def __init__(self, radius=(1, 3), tags=("image",), p=0.5):
        self.p = p
        self.tags = tags
        self.radius = radius

    def get_params(self):
        return {"sigmaX": self.radius[0] + random.random() * (self.radius[1] - self.radius[1])}

    def __call__(self, sample_dict):
        params = self.get_params()
        if random.random() > self.p:
            for tag in self.tags:
                sample_dict[tag] = cv2.GaussianBlur(sample_dict[tag].copy(), None, **params)
        return sample_dict


def augment(sample):
    return TorchvisionCompose(
        [
            ColorJitterCV(brightness=0.8, contrast=0.1, gamma=0.2, temp=0.8, p=0.75),
            RandomGaussianBlur(),
            RandomHorizontalFlip(),
        ]
    )(sample)

In [ ]:
backbone = ViT("B_16", pretrained=True)
IMG_SIZE = backbone.image_size
IMG_SIZE = IMG_SIZE[0] if isinstance(IMG_SIZE, (tuple, list)) else IMG_SIZE
print(f"image size: {IMG_SIZE}")

In [ ]:
os.chdir(DATA_DIR)
train_features = pd.read_csv("train_features.csv", index_col="id")
test_features = pd.read_csv("test_features.csv", index_col="id")
train_labels = pd.read_csv("train_labels.csv", index_col="id")
species_labels = sorted(train_labels.columns.unique())

In [ ]:
y = train_labels.sample(frac=FRAC, random_state=SEED)
x = train_features.loc[y.index].filepath.to_frame()
sites = train_features.loc[y.index, "site"]
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
train_idx, eval_idx = next(gss.split(x, y, groups=sites))
x_train, x_eval = x.iloc[train_idx], x.iloc[eval_idx]
y_train, y_eval = y.iloc[train_idx], y.iloc[eval_idx]

In [ ]:
class ImagesDataset(Dataset):
    def __init__(self, x_df, y_df=None, mode="eval"):
        self.data = x_df
        self.label = y_df
        self.mode = mode

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img = cv2.imread(self.data.iloc[idx]["filepath"])  # BGR uint8
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        sample = {"image": img}
        if self.mode == "train":
            sample = augment(sample)
        img = cv2.cvtColor(np.ascontiguousarray(sample["image"]), cv2.COLOR_BGR2RGB)
        img = img.transpose(2, 0, 1)
        img = torch.from_numpy(img.copy()).float() / 255.0
        img = (img - NORM_MEAN) / NORM_STD
        out = {"image_id": self.data.index[idx], "image": img}
        if self.label is not None:
            out["label"] = torch.tensor(self.label.iloc[idx].values, dtype=torch.float)
        return out

In [ ]:
train_dataset = ImagesDataset(x_train, y_train, mode="train" if AUGMENT else "eval")
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
eval_dataset = ImagesDataset(x_eval, y_eval, mode="eval")
eval_dataloader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True)

In [ ]:
model = LoRA_ViT(backbone, r=RANK, num_classes=NUM_CLASSES).to(device)
counts = y_train[species_labels].sum().values
weights = torch.tensor(counts.sum() / (len(counts) * counts), dtype=torch.float, device=device)
criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=LABEL_SMOOTHING)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=WEIGHT_DECAY)

In [ ]:
def evaluate():
    model.eval()
    parts = []
    with torch.no_grad():
        for batch in eval_dataloader:
            preds = nn.functional.softmax(model(batch["image"].to(device)), dim=1)
            parts.append(pd.DataFrame(preds.cpu().numpy(), index=batch["image_id"], columns=species_labels))
    preds_df = pd.concat(parts).loc[y_eval.index]
    ll = log_loss(y_eval.idxmax(axis=1), preds_df[species_labels], labels=species_labels)
    return ll, preds_df

In [ ]:
best_ll = float("inf")
for epoch in range(1, EPOCHS + 1):
    model.train()
    for batch in tqdm(train_dataloader, total=len(train_dataloader)):
        optimizer.zero_grad()
        loss = criterion(model(batch["image"].to(device)), batch["label"].to(device))
        loss.backward()
        optimizer.step()
    ll, _ = evaluate()
    print(f"epoch {epoch:2d}  train_loss {loss.item():.4f}  eval_logloss {ll:.4f}")
    if ll < best_ll:
        best_ll = ll
        torch.save(model.state_dict(), "best.pth")
print(f"best eval log loss: {best_ll:.4f}")

In [ ]:
model.load_state_dict(torch.load("best.pth"))
_, eval_preds_df = evaluate()
eval_true = y_eval.idxmax(axis=1)
eval_predictions = eval_preds_df.idxmax(axis=1)
print(f"best eval accuracy: {(eval_predictions == eval_true).mean():.4f}")
print("per-class accuracy:")
for sp in species_labels:
    mask = eval_true == sp
    if mask.sum():
        print(f"  {sp:18s} {(eval_predictions[mask] == eval_true[mask]).mean():.3f}  (n={mask.sum()})")